# Notebook 1 — Ingesta de Datos con MongoDB

**Ernesto Investing AI  — Integración (iDeSo)**


Este notebook descarga datos OHLCV reales de los 5 tickers mineros desde Yahoo
Finance, calcula indicadores técnicos y los persiste en MongoDB Atlas en la
colección **`precios_ohlcv`**, que luego leerá el Notebook 2 para entrenar el
clasificador SVC.

**Tickers:** FSM, VOLCABC1.LM, ABX.TO, BVN, BHP


## 1. Instalación de librerías

In [1]:
# Instalamos pymongo con soporte SRV (necesario para el URI de Atlas) y yfinance
!pip install -q "pymongo[srv]" yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.6 MB/s eta 0:00:00


## 2. Conexión a MongoDB Atlas

In [2]:
from google.colab import userdata
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Leemos el URI desde los Secrets de Colab (nunca lo escribimos en texto plano)
MONGO_URI = userdata.get('MONGO_URI')

cliente = MongoClient(MONGO_URI, server_api=ServerApi('1'))

# Verificamos que la conexión funcione
try:
    cliente.admin.command('ping')
    print("✅ Conexión exitosa a MongoDB Atlas")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

# Base de datos y colección donde se guardarán los precios
db = cliente['ernesto_investing_ai']
coleccion = db['precios_ohlcv']

✅ Conexión exitosa a MongoDB Atlas


## 3. Descarga de datos OHLCV (Yahoo Finance)

In [3]:
import yfinance as yf
import pandas as pd

# Tickers de empresas mineras a analizar
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

datos_crudos = {}

for ticker in TICKERS:
    print(f"Descargando {ticker}...")
    df = yf.download(ticker, period='1y', auto_adjust=True, progress=False)
    if df.empty:
        print(f"  ⚠️ Sin datos para {ticker}, se omite.")
        continue
    datos_crudos[ticker] = df
    print(f"  ✅ {len(df)} filas descargadas")

print(f"\nTickers descargados con éxito: {list(datos_crudos.keys())}")

Descargando FSM...
  ✅ 251 filas descargadas
Descargando VOLCABC1.LM...
  ✅ 247 filas descargadas
Descargando ABX.TO...
  ✅ 252 filas descargadas
Descargando BVN...
  ✅ 251 filas descargadas
Descargando BHP...
  ✅ 251 filas descargadas

Tickers descargados con éxito: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']


## 4. Corrección del MultiIndex de yfinance

In [4]:
# yfinance devuelve columnas como MultiIndex (Price, Ticker), incluso al
# descargar un solo ticker. Aplanamos quedándonos con el primer nivel
# (Open, High, Low, Close, Volume).

datos_limpios = {}

for ticker, df in datos_crudos.items():
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index.name = 'Fecha'
    datos_limpios[ticker] = df

# Verificación rápida de la estructura ya aplanada
primer_ticker = list(datos_limpios.keys())[0]
print(f"Columnas de {primer_ticker}: {list(datos_limpios[primer_ticker].columns)}")
datos_limpios[primer_ticker].head()

Columnas de FSM: ['Close', 'High', 'Low', 'Open', 'Volume']


Price,Close,High,Low,Open,Volume
Fecha,,,,,
2025-07-03,6.61,6.67,6.51,6.55,8507200
2025-07-07,6.78,6.80,6.42,6.54,14423900
2025-07-08,6.26,6.80,6.24,6.75,14947500
2025-07-09,6.55,6.62,6.23,6.24,16752800
2025-07-10,6.66,6.70,6.49,6.59,13587100


## 5. Cálculo de indicadores técnicos

In [5]:
def calcular_rsi(serie_close, periodo=14):
    """Calcula el RSI (Relative Strength Index) de N periodos
    usando el promedio móvil simple de ganancias y pérdidas."""
    delta = serie_close.diff()
    ganancia = delta.where(delta > 0, 0.0)
    perdida = -delta.where(delta < 0, 0.0)

    media_ganancia = ganancia.rolling(window=periodo, min_periods=periodo).mean()
    media_perdida = perdida.rolling(window=periodo, min_periods=periodo).mean()

    rs = media_ganancia / media_perdida
    rsi = 100 - (100 / (1 + rs))
    return rsi

for ticker, df in datos_limpios.items():
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['RSI_14'] = calcular_rsi(df['Close'], periodo=14)
    print(f"{ticker}: indicadores calculados ({df.shape[1]} columnas totales)")

FSM: indicadores calculados (10 columnas totales)
VOLCABC1.LM: indicadores calculados (10 columnas totales)
ABX.TO: indicadores calculados (10 columnas totales)
BVN: indicadores calculados (10 columnas totales)
BHP: indicadores calculados (10 columnas totales)


## 6. Preparación de documentos y guardado en MongoDB

In [6]:
import numpy as np
import math

def fila_a_documento(ticker, fecha, fila):
    """Convierte una fila del DataFrame en un documento JSON-válido para MongoDB.
    Los indicadores como SMA_50 generan NaN en los primeros registros
    (no hay suficiente historial); los reemplazamos por None porque
    NaN no es un valor JSON válido."""
    doc = {
        'ticker': ticker,
        'fecha': fecha.strftime('%Y-%m-%d'),
    }
    for columna, valor in fila.items():
        if isinstance(valor, (float, np.floating)) and math.isnan(valor):
            doc[columna] = None
        elif isinstance(valor, np.integer):
            doc[columna] = int(valor)
        elif isinstance(valor, np.floating):
            doc[columna] = float(valor)
        else:
            doc[columna] = valor
    return doc

documentos = []
for ticker, df in datos_limpios.items():
    for fecha, fila in df.iterrows():
        documentos.append(fila_a_documento(ticker, fecha, fila))

print(f"Total de documentos preparados: {len(documentos)}")

Total de documentos preparados: 1252


In [7]:
# Antes de insertar, borramos datos previos de estos mismos tickers
# para evitar duplicados si el notebook se vuelve a ejecutar
tickers_procesados = list(datos_limpios.keys())
coleccion.delete_many({'ticker': {'$in': tickers_procesados}})

if documentos:
    resultado = coleccion.insert_many(documentos)
    print(f"✅ {len(resultado.inserted_ids)} documentos insertados en 'precios_ohlcv'")
else:
    print("⚠️ No hay documentos para insertar")

✅ 1252 documentos insertados en 'precios_ohlcv'


## 7. Verificación final

In [8]:
import pprint

print("=== Verificación de la colección precios_ohlcv ===\n")

total = coleccion.count_documents({})
print(f"Total de documentos en la colección: {total}\n")

for ticker in TICKERS:
    cantidad = coleccion.count_documents({'ticker': ticker})
    estado = "✅" if cantidad > 0 else "⚠️"
    print(f"  {estado} {ticker}: {cantidad} documentos")

print("\n--- Ejemplo de documento guardado (más reciente) ---")
ejemplo = coleccion.find_one(
    {'ticker': tickers_procesados[0]},
    sort=[('fecha', -1)]
)
pprint.pprint(ejemplo)

=== Verificación de la colección precios_ohlcv ===

Total de documentos en la colección: 1252

  ✅ FSM: 251 documentos
  ✅ VOLCABC1.LM: 247 documentos
  ✅ ABX.TO: 252 documentos
  ✅ BVN: 251 documentos
  ✅ BHP: 251 documentos

--- Ejemplo de documento guardado (más reciente) ---
{'Close': 8.720000267028809,
 'EMA_12': 8.66930803794649,
 'EMA_26': 8.957302154075867,
 'High': 8.829999923706055,
 'Low': 8.470000267028809,
 'Open': 8.75,
 'RSI_14': 51.661640907852,
 'SMA_20': 8.787500047683716,
 'SMA_50': 9.430400047302246,
 'Volume': 4956400.0,
 '_id': ObjectId('6a4827c52d9c021695e36fcd'),
 'fecha': '2026-07-02',
 'ticker': 'FSM'}
